# NumPy Interview Questions — Advanced Array Toolkit (Week 6)

This notebook collects the NumPy topics that interviewers love to probe and that day-to-day array work keeps demanding. It is not a beginner's tour; it assumes you are comfortable with `np.array`, shapes, and basic slicing, and it goes straight at the functions that separate a confident NumPy user from a hesitant one.

The journey starts with **meshgrids and 3-D plotting**. You generate coordinate grids with `np.meshgrid`, evaluate a function like `3*log(x) + 2*y` across the whole grid at once, and render the result as a scatter, then a surface, first with matplotlib and then with Plotly's interactive `go.Surface` — the standard recipe for visualising a two-variable function.

Next comes **randomness**, a topic every data scientist must control carefully. You meet `np.random.random` for arbitrary shapes, `np.random.seed(0)` for *reproducible* results (the same seed always yields the same `45, 48, 65...` array, which is exactly why seeded experiments are trustworthy), `np.random.shuffle` to permute an array in place, and `np.random.choice(a, 3, replace=False)` to draw a sample **without replacement**.

Third comes **images as arrays** — arguably the most eye-opening section. A PNG loaded with OpenCV (`cv2.imread`) is just an array of shape `(1080, 1920, 3)`, one byte per RGB channel. That one fact unlocks image processing with ordinary NumPy: flip horizontally (`np.flip(img, axis=1)`), transform with element-wise arithmetic (`255 - img` for a negative), trim with slicing (`img[100:900, 50:900, :]`), fade by clipping (`np.clip(img, 0, 100)`), and view brightness distributions with `plt.hist(img.flatten(), bins=255)`.

Finally the notebook tours specialised array features: **structured arrays** (a single `np.dtype` describing named fields of different types — a lightweight record table, saved and reloaded with `np.save`), and a grab-bag of utility functions — `np.swapaxes`, `np.random.uniform`, `np.count_nonzero`, `np.tile`, `np.repeat`, and `np.allclose`. Each one is demonstrated with a short, runnable example so you can see the exact shape and dtype produced. Work through the cells in order and make each output a memory anchor.

## 0. Setup

The first cell creates a `linspace` to prepare for meshgrids.

In [25]:
# plotting 3D graphs
# meshgrids
a = np.linspace(-10,9,20)
b = np.linspace(-10,9,20)
b

array([-10.,  -9.,  -8.,  -7.,  -6.,  -5.,  -4.,  -3.,  -2.,  -1.,   0.,
         1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.])

# 1. Meshgrids and 3-D Visualisation

A meshgrid turns two 1-D coordinate axes into two 2-D matrices so that every combination of an x and a y value can be evaluated and plotted together.

In [26]:
xx,yy = np.meshgrid(a,b)
yy

array([[-10., -10., -10., -10., -10., -10., -10., -10., -10., -10., -10.,
        -10., -10., -10., -10., -10., -10., -10., -10., -10.],
       [ -9.,  -9.,  -9.,  -9.,  -9.,  -9.,  -9.,  -9.,  -9.,  -9.,  -9.,
         -9.,  -9.,  -9.,  -9.,  -9.,  -9.,  -9.,  -9.,  -9.],
       [ -8.,  -8.,  -8.,  -8.,  -8.,  -8.,  -8.,  -8.,  -8.,  -8.,  -8.,
         -8.,  -8.,  -8.,  -8.,  -8.,  -8.,  -8.,  -8.,  -8.],
       [ -7.,  -7.,  -7.,  -7.,  -7.,  -7.,  -7.,  -7.,  -7.,  -7.,  -7.,
         -7.,  -7.,  -7.,  -7.,  -7.,  -7.,  -7.,  -7.,  -7.],
       [ -6.,  -6.,  -6.,  -6.,  -6.,  -6.,  -6.,  -6.,  -6.,  -6.,  -6.,
         -6.,  -6.,  -6.,  -6.,  -6.,  -6.,  -6.,  -6.,  -6.],
       [ -5.,  -5.,  -5.,  -5.,  -5.,  -5.,  -5.,  -5.,  -5.,  -5.,  -5.,
         -5.,  -5.,  -5.,  -5.,  -5.,  -5.,  -5.,  -5.,  -5.],
       [ -4.,  -4.,  -4.,  -4.,  -4.,  -4.,  -4.,  -4.,  -4.,  -4.,  -4.,
         -4.,  -4.,  -4.,  -4.,  -4.,  -4.,  -4.,  -4.,  -4.],
       [ -3.,  -3.,  -3.,  -3.,  -3.,  -3

## 1.1 np.meshgrid — from one axis to a full coordinate grid

`a = np.linspace(-10, 9, 20)` produces 20 evenly spaced numbers; the same is done for `b`. Calling `xx, yy = np.meshgrid(a, b)` then returns two 20x20 arrays: `xx` holds the x-coordinate of every grid point, `yy` the matching y-coordinate. Reading the printed `yy` confirms the mechanism — every row of `yy` is a constant value repeated across the columns, because y varies *down* the rows, while `xx` varies across the columns. Flatten the pair and you have the coordinates of all 400 grid points.

`plt.scatter(xx, yy)` plots these points and the result is a perfect lattice — the visual evidence that `meshgrid` filled a rectangular domain uniformly. This is precisely the coordinate scaffold needed for evaluating any function of two variables over a rectangle, and it is the standard preparation for 3-D surface plots, contour plots, and heatmaps. The shape rule to remember: if `a` has length `ny` and `b` has length `nx`, `meshgrid` by default returns arrays of shape `(len(b), len(a))`, and you can switch the orientation with `indexing='ij'` vs `'xy'` — a frequent source of plot-orientation bugs.

In [27]:
import matplotlib.pyplot as plt
plt.scatter(xx,yy)

<Figure size 432x288 with 1 Axes>

In [34]:
def func(x,y):
  return 3*np.log(x) + 2*y

zz = func(xx,yy) 
zz

<ipython-input-34-78e1ed4898ab>:2: RuntimeWarning:

divide by zero encountered in log

<ipython-input-34-78e1ed4898ab>:2: RuntimeWarning:

invalid value encountered in log



array([[         nan,          nan,          nan,          nan,
                 nan,          nan,          nan,          nan,
                 nan,          nan,         -inf, -20.        ,
        -17.92055846, -16.70416313, -15.84111692, -15.17168626,
        -14.62472159, -14.16226955, -13.76167537, -13.40832627],
       [         nan,          nan,          nan,          nan,
                 nan,          nan,          nan,          nan,
                 nan,          nan,         -inf, -18.        ,
        -15.92055846, -14.70416313, -13.84111692, -13.17168626,
        -12.62472159, -12.16226955, -11.76167537, -11.40832627],
       [         nan,          nan,          nan,          nan,
                 nan,          nan,          nan,          nan,
                 nan,          nan,         -inf, -16.        ,
        -13.92055846, -12.70416313, -11.84111692, -11.17168626,
        -10.62472159, -10.16226955,  -9.76167537,  -9.40832627],
       [         nan,          nan,  

## 1.2 Evaluating and plotting a surface in 3-D

With the grid ready, `def func(x, y): return 3*np.log(x) + 2*y` is applied to the **whole grid** at once: `zz = func(xx, yy)`. Because `xx` and `yy` are arrays, the function is evaluated element-wise, giving a 20x20 height map. NumPy emits `RuntimeWarning: divide by zero encountered in log` and `invalid value encountered in log` where `x` is zero or negative — `np.log` of a non-positive number is `-inf`/`nan` — and those cells become `nan` in `zz`. That warning is a useful lesson: vectorised maths propagates edge cases silently into `nan` rather than raising, so always sanity-check inputs for logarithms and square roots.

Plotting follows two routes. `plt.scatter(xx, yy)` shows the sampling grid; the interactive version uses Plotly:
```python
import plotly.express as px
import plotly.graph_objects as go
fig = px.scatter_3d()
fig.add_trace(go.Surface(x=xx, y=yy, z=zz))
fig.show()
```
`go.Surface` builds a coloured surface through all the grid points, letting you rotate and zoom to inspect the shape — far more informative than a 2-D scatter when the function is genuinely three-dimensional. The recipe 'make a grid, evaluate a function on it, hand the three matrices to a surface plot' recurs throughout scientific computing.

In [35]:
import plotly.express as px
import plotly.graph_objects as go

fig = px.scatter_3d()

fig.add_trace(go.Surface(x=xx,y=yy,z=zz))

fig.show()

# 2. Working with Random Numbers

Randomness needs discipline in data science: reproducible experiments demand settings, and simulation demands a command of the generator's API.

In [2]:
# working with random
# randint
# seed
# shuffle
# choice

## 2.1 random, seed, shuffle, choice

The `np.random` submodule covers the four verbs in the section heading. `np.random.random((2,3,2))` draws uniform values in `[0, 1)` in any requested shape — here two 3x2 blocks. `np.random.randint(1, 100, 12)` draws 12 integers in `[1, 100)` (the high bound is exclusive), and `.reshape(3, 4)` arranges them into a matrix.

The important cell is `np.random.seed(0)`. Run `np.random.seed(0)` followed by `np.random.randint(1,100,12).reshape(3,4)` and you get `[[45, 48, 65, 68], ...]`; run the identical pair again and you get the **same numbers**. That is the entire point of seeding: it fixes the generator's starting state so that shuffles, splits, and random initialisations are reproducible across runs and machines. In machine learning this is why you set seeds before data shuffling or weight initialisation — results must be repeatable for debugging and fair comparison.

`np.random.shuffle(a)` permutes an array **in place** (it returns `None`; the variable `a` is changed, as the following cell shows). `np.random.choice(a, 3, replace=False)` samples 3 elements **without replacement** — crucial when you want distinct items, such as train/test splits or lottery draws; `replace=True` (the default) would allow duplicates. Combined, these four functions are the workhorses of simulation, bootstrap sampling, and every random-split utility you will ever write.

In [37]:
np.random.random((2,3,2))

array([[[0.28969975, 0.30904037],
        [0.02229412, 0.08411571],
        [0.34225695, 0.87044578]],

       [[0.3088764 , 0.55506361],
        [0.95240073, 0.44318119],
        [0.28857773, 0.17184448]]])

In [41]:
np.random.seed(0)
np.random.randint(1,100,12).reshape(3,4)

array([[45, 48, 65, 68],
       [68, 10, 84, 22],
       [37, 88, 71, 89]])

In [42]:
np.random.seed(0)
np.random.randint(1,100,12).reshape(3,4)

array([[45, 48, 65, 68],
       [68, 10, 84, 22],
       [37, 88, 71, 89]])

In [47]:
np.random.seed(0)
np.random.randint(1,100,12).reshape(3,4)

array([[45, 48, 65, 68],
       [68, 10, 84, 22],
       [37, 88, 71, 89]])

In [48]:
a = np.array([12,41,33,67,89,100])
print(a)

[ 12  41  33  67  89 100]


In [49]:
np.random.shuffle(a)

In [50]:
a

array([ 67, 100,  41,  33,  89,  12])

In [59]:
np.random.choice(a,3,replace=False)

array([ 33, 100,  89])

# 3. Working with Images

An image is nothing but an array of numbers. Once you internalise that, the entire NumPy toolbox becomes an image-editing suite.

In [61]:
# working with images
import cv2

## 3.1 An image is just an ndarray

`img = cv2.imread('screenshot1.png')` loads a PNG into a NumPy array; `img.shape` reports `(1080, 1920, 3)` — height 1080, width 1920, and 3 colour channels (Blue, Green, Red in OpenCV's ordering, which is why plots sometimes look colour-swapped unless you convert). Each pixel is thus a triplet of integers 0–255, and the whole image is just a big array you can slice and compute on.

`plt.imshow(np.flip(img, axis=1))` displays the image flipped horizontally — `np.flip` reverses the order of elements along the chosen axis, and axis 1 is the width, so the image mirrors left-right. The standalone flip demo makes the operation plain on a small array: flipping `[[0,1,2],[3,4,5]]` gives `[[5,4,3],[2,1,0]]` (another slice-style view of the same thing).

More transformations follow:
- `np.clip(img, 0, 100)` clamps every channel value to the range 0–100, darkening/fading the image while preserving structure.
- `255 - img` inverts brightness — the photographic **negative** — and shows how plainly arithmetic acts on pixels.
- `img[100:900, 50:900, :]` **trims** the image by row/column slicing, keeping everything between the chosen bounds.
- `plt.hist(img.flatten(), bins=255)` flattens all channel values and plots their frequency distribution; the shape of this histogram tells you whether the image is dark, bright, or high-contrast. `flatten()` is the bridge from a 3-D image (or any ndarray) to a 1-D sample suitable for statistics.

Each of these is one expression — remember the shape `(H, W, 3)` and axis conventions, and image processing becomes ordinary array maths.

In [64]:
# read the image
img = cv2.imread('/content/screenshot1.png')

In [65]:
# show array -> shape
img.shape

(1080, 1920, 3)

In [71]:
# show image
plt.imshow(np.flip(img,axis=1))

<Figure size 432x288 with 1 Axes>

In [67]:
# flip
a = np.arange(6).reshape(2,3)
a

array([[0, 1, 2],
       [3, 4, 5]])

In [68]:
np.flip(a)

array([[5, 4, 3],
       [2, 1, 0]])

In [76]:
# clip -> fade
plt.imshow(np.clip(img,0,100))

<Figure size 432x288 with 1 Axes>

In [77]:
# negative
plt.imshow(255 - img)

<Figure size 432x288 with 1 Axes>

In [81]:
# trim
plt.imshow(img[100:900,50:900,:])

<Figure size 432x288 with 1 Axes>

In [83]:
# plot histogram
plt.hist(img.flatten(),bins=255)

(array([205709.,  26478.,  22496.,  24156.,  26061.,  27144.,  37194.,
         42868.,  48354.,  42823.,  51278.,  53786.,  44132., 185308.,
        135016., 137699., 167025., 131062., 125666., 123704., 163884.,
        120953., 122947., 136906., 151304., 133090., 132126., 123105.,
         88560., 114552., 108528.,  74169.,  92631.,  64596.,  76882.,
         53629.,  61145.,  39742.,  43245.,  22674.,   6509.,   3723.,
          3422.,   3473.,   3372.,   3382.,  34249.,   3205.,   3491.,
          3358.,   3831.,  71214.,   4049.,  33943.,   4043.,   3985.,
          4147.,   4055.,   4242.,   4633., 141864.,   3722.,   4225.,
          4836., 141839.,   4054.,   5356.,  50107.,   6289.,  35701.,
          3326.,   2543.,   2480.,   4497.,   2555.,   2413.,   2538.,
          2518.,   2459.,  46022.,   2539.,   2399.,   2533.,   3118.,
          2300.,   2474.,   2294.,   2465.,   2497.,   2419.,   2412.,
          2216.,   3462.,   2351.,   2293.,   2100.,   2290.,   2113.,
      

<Figure size 432x288 with 1 Axes>

In [ ]:
# More manipulations
# https://www.analyticsvidhya.com/blog/2021/05/image-processing-using-numpy-with-practical-implementation-and-code/

# 4. Structured Arrays

NumPy arrays normally hold a single dtype. A structured array relaxes that: one array whose elements have several named fields of different types, like a table row.

In [84]:
# structured arrays
a = np.array([1,'hello',True,1.5])
a

array(['1', 'hello', 'True', '1.5'], dtype='<U32')

In [91]:
a[0] / 100

TypeError: ignored

## 4.1 One array, many typed fields

The motivating cell shows the problem: `np.array([1, 'hello', True, 1.5])` coerces everything to strings and stores the result with `dtype='<U32'` — a single common type. You lose the numeric nature of `1` and the boolean nature of `True`. Worse, `a[0] / 100` cannot work, because `a[0]` is now the *string* `'1'`: this is the price of a homogeneous array dtype.

A **structured dtype** fixes this. `np.dtype([('name','U20'), ('iq', np.int32), ('cgpa', np.float64), ('placed','U20')])` defines a record with four named, individually typed fields — the equivalent of a table's schema in miniature. Building `stu = np.array([('nitish',100,6.66,'Yes'), ...], dtype=dt)` then stores heterogeneous data *without coercion*: each tuple is a record, each field keeps its declared type. `stu['placed']` extracts just that column (`array(['Yes','Yes','No'], ...)`) — the same column-extraction syntax as a Pandas DataFrame. `dt` itself prints the resolved dtype, revealing the byte-level types (`<U20` strings, `<i4` int32, `<f8` float64).

Finally `np.save('student.npy', stu)` persists the whole structured array to disk in NumPy's `.npy` format, reloadable with `np.load` keeping names and dtypes intact. Structured arrays are lighter than DataFrames and ideal when you need fast, homogeneous-record storage without the Pandas overhead.

In [101]:
# name,iq,cgpa,placed

dt = np.dtype(
    [
        ('name','U20'),
        ('iq',np.int32),
        ('cgpa',np.float64),
        ('placed','U20')
    ]
)

In [102]:
dt

dtype([('name', '<U20'), ('iq', '<i4'), ('cgpa', '<f8'), ('placed', '<U20')])

In [103]:
stu = np.array(
    [
        ('nitish',100,6.66,'Yes'),
        ('ankit',120,8.9,'Yes'),
        ('rahul',80,7.3,'No')
    ],dtype=dt
)

stu

array([('nitish', 100, 6.66, 'Yes'), ('ankit', 120, 8.9 , 'Yes'),
       ('rahul',  80, 7.3 , 'No')],
      dtype=[('name', '<U20'), ('iq', '<i4'), ('cgpa', '<f8'), ('placed', '<U20')])

In [113]:
stu['placed']

array(['Yes', 'Yes', 'No'], dtype='<U20')

In [114]:
# save and load numpy objects
np.save('student.npy',stu)

# 5. Remaining Utility Functions

A final sweep through functions that appear constantly in code review and interviews. Each subsection below has a short demo.

In [ ]:
# remaining functions
# --> np.swapaxes
# --> np.uniform
# --> np.count_nonzero
# --> np.tile
# --> np.repeat
# --> np.allclose

###`np.swapaxes(arr, axis1, axis2)`
```
Interchange two axes of an array.

Syntax : numpy.swapaxes(arr, axis1, axis2)
Parameters :
arr : [array_like] input array.
axis1 : [int] First axis.
axis2 : [int] Second axis.
Return : [ndarray]
```

## 5.1 np.swapaxes — interchange two axes

`np.swapaxes(arr, axis1, axis2)` exchanges two axes of an array. The example takes `x` of shape `(2, 3)` — `[[1,2,3],[4,5,6]]` — and calls `np.swapaxes(x, 0, 1)`, producing the shape `(3, 2)` array `[[1,4],[2,5],[3,6]]`. Rows became columns and vice versa; the returned array is a **view**, so it shares memory with the original and no data is copied (important for large arrays).

The crucial note that follows the demo distinguishes swapping from **reshaping**. `x.reshape(3, 2)` yields `[[1,2],[3,4],[5,6]]` — the same shape `(3, 2)` but *completely different element order*, because reshape only reinterprets the flat buffer without moving elements, while `swapaxes` moves the axes themselves. For typical 2-D cases `swapaxes` matches transpose, and for higher dimensions you can chain swaps or use `np.transpose(arr, axes)`. Getting the distinction right prevents one of the most common subtle bugs in tensor manipulation: values silently sliding across axes.

In [ ]:
#Example 
x = np.array([[1,2,3],[4,5,6]])
print(x)
print(x.shape)
print("Swapped")
x_swapped = np.swapaxes(x,0,1)
print(x_swapped)
print(x_swapped.shape)


[[1 2 3]
 [4 5 6]]
(2, 3)
Swapped
[[1 4]
 [2 5]
 [3 6]]
(3, 2)


Note: It is not same as reshaping.

In [ ]:
x_reshaped = x.reshape(3,2)
print(x_reshaped)

[[1 2]
 [3 4]
 [5 6]]


###`numpy.random.uniform(low=0.0, high=1.0, size=None)`
Draw samples from a uniform distribution in rangge [low - high); high not included.
https://numpy.org/doc/stable/reference/random/generated/numpy.random.uniform.html
```
Syntax : numpy.random.uniform(low, high, size=None)
            low -> lower bound of sample; default value is 0
            high -> uper bound of sample; defalut value is 1.0
            size -> shape of the desired sample. If the given shape is, e.g., (m, n, k), then m * n * k samples are drawn.

Return : Return the random samples as numpy array.
```
When ever we need to test our model on uniform data and we might not get truly uniform data in real scenario, we can use this function to randomly generate data for us.

## 5.2 np.random.uniform — samples from a flat distribution

`np.random.uniform(low=0.0, high=1.0, size=None)` returns random floats drawn uniformly from the half-open interval `[low, high)` — `high` itself is never produced. The first example, `np.random.uniform(0, 11, 10)`, returns ten floats like `7.55, 8.34, 3.47, ...` scattered through the range. The second draws 1000 samples from `[-5, 10)` and plots them with `plt.hist(..., bins=30, density=True)`, producing the signature flat-topped histogram that proves the density is constant across the interval.

Why reach for this function? Real-world data is rarely perfectly uniform, so when you need to test a model, a statistical routine, or a visualisation under clean uniform conditions, this generator manufactures exactly that input on demand. It is also the primitive behind sampling a continuous interval — for example, choosing a random threshold or jittering points. Contrast it with `np.random.random`, which is simply `uniform(0, 1)` in disguise, and with `np.random.normal` when you need a bell-shaped rather than a flat distribution. Remember the half-open interval: `uniform(0, 1)` can return 0.0 but never 1.0.

In [ ]:
#Example:
uniform = np.random.uniform(0, 11, 10)
print(uniform)

[ 7.54634884  8.34285907  3.46529107  8.5453016  10.4852508   4.64808768
 10.70041412  3.27949425  0.10020655  8.75951609]


In [ ]:
#Example:
import matplotlib.pyplot as plt
  
# Using uniform() method
uniform = np.random.uniform(-5, 10, 1000)

plt.hist(uniform, bins = 30, density = True)
plt.show()

<Figure size 432x288 with 1 Axes>

###`np.count_nonzero(arr, axis=None)`
This function counts the number of non-zero values in the array
https://numpy.org/doc/stable/reference/generated/numpy.count_nonzero.html

```
Syntax : numpy.count_nonzero(arr, axis=None)

Parameters :
    arr : [array_like] The array for which to count non-zeros.

    axis : [int or tuple, optional] Axis or tuple of axes along which to count non-zeros. Default is None, meaning that non-zeros will be counted along a flattened version of arr.

    keepdims : [bool, optional] If this is set to True, the axes that are counted are left in the result as dimensions with size one.

Return : [int or array of int] Number of non-zero values in the array along a given axis. Otherwise, the total number of non-zero values in the array is returned.
```


## 5.3 np.count_nonzero — tally truth-like entries

`np.count_nonzero(arr, axis=None)` counts elements that are non-zero (equivalently, truthy). Called on the 2x4 array `[[0,1,7,0],[3,0,2,19]]` with no axis, it flattens and returns `5` — the five non-zero values. With `axis=0` it counts **down each column**: `[1, 1, 2, 1]` (column 0 has one non-zero, column 2 has two, etc.). With `axis=1` it counts **across each row**: `[2, 3]` (row 0 has two non-zeros, row 1 has three). The `keepdims` option, when `True`, retains the reduced axes as length-1 dimensions, which helps broadcasting the result back against the original array.

This function is the standard way to count boolean conditions after masking: `np.count_nonzero(arr > 5)` counts values above five, and `np.count_nonzero(arr == 0)` counts zeros or missing-coded entries. It is generally faster than `(arr != 0).sum()` because it avoids building an intermediate boolean array, so it turns up in tight loops and large-data pipelines where every allocation counts.

In [97]:
#Example

a = np.array([[0, 1, 7, 0],
              [3, 0, 2, 19]])

print(np.count_nonzero(a))

print(np.count_nonzero(a, axis=0))

print(np.count_nonzero(a, axis=1))


5
[1 1 2 1]
[2 3]


###`np.tile(A, reps)`
Construct an array by repeating `A` the number of times given by `reps`.
If reps has length `d`, the result will have dimension of `max(d, A.ndim)`.
```
Parameters:

    A: array_like
        The input array.

    reps: array_like
        The number of repetitions of A along each axis.

Returns
    c: ndarray
        The tiled output array.
```
https://numpy.org/doc/stable/reference/generated/numpy.tile.html

## 5.4 np.tile — repeat the whole array

`np.tile(A, reps)` constructs a new array by repeating `A` the number of times given in `reps`. Starting with `a = np.array([0, 1, 2])`, `np.tile(a, 2)` produces `[0, 1, 2, 0, 1, 2]` — the entire array duplicated twice along the (new) last axis.

Passing a tuple of reps controls repetition per axis. `np.tile(a, (2, 2))` first promotes `a` to 2-D and then repeats it 2 times *down* (axis 0) and 2 times *across* (axis 1), yielding a 2x6 array of two identical `[0,1,2,0,1,2]` rows. `np.tile(a, (2, 3))` keeps 2 rows but repeats the content 3 times across, giving 2x9. The rule of thumb: `reps` with length `d` produces an output of dimension `max(d, A.ndim)`, and any missing leading axes are treated as 1.

The mental model is *tiling a floor* — the pattern `A` is stamped repeatedly in a grid — as opposed to `np.repeat`, which duplicates individual *elements*. Confusing the two is a classic interview trap, so note the difference now: `tile` repeats the block, `repeat` repeats each value.

In [ ]:
# np.tile - Example
a = np.array([0, 1, 2])
print(a)
print("Tiled")
print(np.tile(a, 2))
# Reps is given as 2 so whole array will get repeted 2 times


[0 1 2]
Tiled
[0 1 2 0 1 2]


In [ ]:
np.tile(a, (2, 2))
# Reps is given as (2, 2) 
# means along axis-0, 2 time repetition and 
# along axis-1, 2 times repetition
# Axis-0 downward along the rows
# Axis -1 rightward along columns -> or inside each rows.

array([[0, 1, 2, 0, 1, 2],
       [0, 1, 2, 0, 1, 2]])

In [ ]:
np.tile(a, (2, 3)) # Along axis-1 3 times repetition

array([[0, 1, 2, 0, 1, 2, 0, 1, 2],
       [0, 1, 2, 0, 1, 2, 0, 1, 2]])

###`np.repeat(a, repeats, axis=None)`
Repeat elements of an array. `repeats` parameter says no of time to repeat

```
Parameters: 
            a: array_like
                    Input array.

            repeats: int or array of ints
                    The number of repetitions for each element. repeats is broadcasted to fit the shape of the given axis.

            axis: int, optional
                    The axis along which to repeat values. By default, use the flattened input array, and return a flat output array.

Returns: 
    repeated_array: ndarray
                    Output array which has the same shape as a, except along the given axis.
```
https://numpy.org/doc/stable/reference/generated/numpy.repeat.html

## 5.5 np.repeat — duplicate individual elements

`np.repeat(a, repeats, axis=None)` repeats *each element* `repeats` times. On `x = [[1,2],[3,4]]`, `np.repeat(x, 2)` flattens first (axis is `None` by default) and repeats every value twice: `[1, 1, 2, 2, 3, 3, 4, 4]`.

Supplying an axis changes the shape dramatically, because repetition happens along that axis and the array grows in that dimension:
- `np.repeat(x, 3, axis=1)` repeats each value 3 times *within the rows*, so columns multiply: `[[1,1,1,2,2,2],[3,3,3,4,4,4]]` (shape 2x6).
- `np.repeat(x, 3, axis=0)` repeats each *row* 3 times, so rows multiply: shape 6x2, with each original row appearing three times.

A key capability beyond `tile` is that `repeats` may itself be an array, letting each element be repeated a different number of times — for example `np.repeat([1,2,3], [1,2,3])` gives `[1,2,2,3,3,3]`. This is exactly how you expand grouped data back into per-item form and how you build weighted samples. The distinction to memorise: `tile` repeats the whole block, `repeat` repeats element-wise along a chosen axis.

In [ ]:
x = np.array([[1,2],[3,4]])
print(x)
print(np.repeat(x, 2)) # Every element is getting repeted 2 times.

[[1 2]
 [3 4]]
[1 1 2 2 3 3 4 4]


In [ ]:
print(x)
print(np.repeat(x, 3, axis=1)) # Alog axis-1 means rightward inside rows/ along columns
# Along axis-1 columns will increase

[[1 2]
 [3 4]]
[[1 1 1 2 2 2]
 [3 3 3 4 4 4]]


In [ ]:
print(np.repeat(x, 3, axis=0)) # Alog axis-0 means downward to rows/ along inside a column
# Along axis-0 rows will increase


[[1 2]
 [1 2]
 [1 2]
 [3 4]
 [3 4]
 [3 4]]


### `np.allclose`
Returns True if two arrays are element-wise equal within a tolerance.

The tolerance values are positive, typically very small numbers. The relative difference `(rtol * abs(b))` and the absolute difference `atol` are added together to compare against the `absolute difference` between `a and b`.

If the following equation is element-wise True, then `allclose` returns `True`.

> `absolute(a - b) <= (atol + rtol * absolute(b))`

```
Syntax : numpy.allclose(arr1, arr2, rtol, atol, equal_nan=False)

Parameters :
    arr1 : [array_like] Input 1st array.
    arr2 : [array_like] Input 2nd array.
    rtol : [float] The relative tolerance parameter.
    atol : [float] The absolute tolerance parameter.
    equal_nan : [bool] Whether to compare NaN’s as equal. 
            If True, NaN’s in arr1 will be considered equal to NaN’s in arr2 in the output array.

Return : [ bool] Returns True if the two arrays are equal within the given tolerance, otherwise it returns False.
```
https://numpy.org/doc/stable/reference/generated/numpy.allclose.html
https://www.geeksforgeeks.org/numpy-allclose-in-python/


## 5.6 np.allclose — equality within tolerance

Floating-point arithmetic is inexact, so `a == b` on genuinely equal computations often returns `False`. `np.allclose(arr1, arr2, rtol, atol, equal_nan=False)` answers the more useful question: are two arrays equal *within a tolerance*? It returns `True` when, element-wise, `absolute(a - b) <= (atol + rtol * absolute(b))`.

The demo makes the tolerance tangible. With `a = [1.1, 1.2, 1.0001]` and `b = [1.0, 1.02, 1.001]`, the absolute differences print as `[0.1, 0.18, 0.0009]`. The default `np.allclose(a, b)` returns `False` because the second difference (0.18) is too large, but `np.allclose(a, b, atol=0.2)` returns `True` — relaxing the absolute tolerance to 0.2 admits all three. `rtol` scales the allowed difference by the magnitude of `b`, which is why allclose behaves sensibly across very large and very small numbers (a relative comparison), while `atol` provides an absolute floor near zero where relative comparison is meaningless.

The final cell addresses `nan`: `np.allclose([1.0, np.nan], [1.0, np.nan])` returns `False` because `nan != nan` by IEEE rules; passing `equal_nan=True` treats matching `nan`s as equal and returns `True`. This function is the correct way to assert numerical results in tests and to compare model outputs, and getting the `atol`/`rtol`/`equal_nan` combination right is what makes such checks robust.

In [ ]:
#np.allclose example
#Comparing - 
a = np.array([1.1, 1.2, 1.0001])
b = np.array([1., 1.02, 1.001])
print(a)
print(b)
print(np.abs(a-b))

print(np.allclose(a,b)) # will return false
print(np.allclose(a,b, atol=0.2)) # will return true, as i incearse the absolute tolerance value


[1.1    1.2    1.0001]
[1.    1.02  1.001]
[0.1    0.18   0.0009]
False
True


In [ ]:
print(np.allclose([1.0, np.nan], [1.0, np.nan])) # Nan will be taken as different

print(np.allclose([1.0, np.nan], [1.0, np.nan], equal_nan=True)) # Nan will be treated as same

False
True


## Summary

Week 6's NumPy interview sweep covered the array operations most likely to appear in technical discussions. **Meshgrid** (`np.meshgrid`) built 2-D coordinate grids from 1-D axes, enabling element-wise evaluation of `3*log(x) + 2*y` and 3-D surface rendering with matplotlib and Plotly's `go.Surface`. **Randomness** was tamed with `np.random.random`, `randint`, and the all-important `np.random.seed(0)` for reproducible results, plus in-place `shuffle` and sampling without replacement via `choice(..., replace=False)`. **Images** were shown to be plain `(H, W, 3)` arrays, which made flipping (`np.flip`), inverting (`255 - img`), clipping/fading (`np.clip`), trimming (slicing), and histogramming (`flatten` + `hist`) ordinary array arithmetic. **Structured arrays** stored heterogeneous, named, typed records in a single ndarray and persisted them with `np.save`. Finally the utility tour distinguished `swapaxes` from `reshape`, drew uniform samples with `np.random.uniform`, counted conditions with `np.count_nonzero`, contrasted block-tiling `np.tile` with element-wise `np.repeat`, and compared floats safely with `np.allclose` — tolerance, `equal_nan`, and all.